# Word2Vec Semantic Neighbourhood Analysis
## Targeted at LLR-Significant Thematic Keywords: Space / Color / Haunting Methods
### Kelly Meng-Yu Chiu | KAIST Culture Technology | 2026

---

**Purpose of this notebook:**  
This notebook trains Word2Vec models on raw ghost story corpora (PTT / Reddit / Theqoo) and queries the **semantic neighbourhood** of keywords that were identified as statistically significant collocates (LLR, FDR-corrected) in the main analysis pipeline. The goal is to enrich the thematic groupings of Space, Color, and Haunting Methods with contextual evidence — i.e., what words appear in similar distributional contexts as these significant collocates?

**Methodological position in the thesis:**  
This analysis serves as a *contextual enrichment layer* for the LLR collocate results. It does NOT replace LLR as the primary statistical analysis. Instead, it provides the semantic neighbourhood evidence used in the Discussion chapter (Ch. 5) to reconstruct narrative structure around gendered ghost depictions.

**Three-platform thematic keyword sets are derived directly from LLR output files:**  
`collocate_LLR_ptt.csv`, `collocate_LLR_theqoo.csv`, `collocate_LLR_reddit.csv`  
Only words with `sig_fdr ≥ *` (i.e., p_fdr < 0.05) are included.


---
## Part 0: Setup & Dependencies

In [ ]:
import re
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')

# Gensim Word2Vec
from gensim.models import Word2Vec

# ── NLP Tokenizers ──
JIEBA_OK = False
SPACY_OK = False
KIWI_OK  = False

try:
    import jieba
    JIEBA_OK = True
    print('jieba loaded.')
except ImportError:
    print('jieba not available — PTT tokenisation will use regex fallback.')

try:
    import spacy
    nlp_en = spacy.load('en_core_web_sm')
    SPACY_OK = True
    print('spaCy en_core_web_sm loaded.')
except Exception:
    print('spaCy not available — Reddit tokenisation will use regex fallback.')

try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    KIWI_OK = True
    print('kiwipiepy loaded.')
except ImportError:
    print('kiwipiepy not available — Theqoo tokenisation will use regex fallback.')

# Font for CJK plots
matplotlib.rcParams['axes.unicode_minus'] = False


--- ## Part 1: Thematic Keywords (Space / Color / Haunting Method)

These keyword sets define thematic categories for each platform.
Each domain (Space, Color, Haunting Method) is organised by **category** within each platform.
Keywords are used to query Word2Vec semantic neighbourhoods.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# THEMATIC KEYWORDS — Space / Color / Haunting Method
# Format: {platform: {category: [keywords]}}
# ─────────────────────────────────────────────────────────────────────────────
SPACE_GROUPS = {
    "ptt": {
        "Indoor (Domestic)": ['房間', '房門','廚房','臥房','阿太房','空房','病房',
                    '客房','房屋','暗房','樓房','新房子','雅房','大房子','書房',
                    '小房','平房','套房','茅房','小書房','睡房','老房子','房子','大樓',
                    '閣樓','儲藏室','和室','休息室','山莊',
                    '紅眠床','大床','病床','彈簧床','床位','床沿','床邊'],
        "Sleep_Dream": ['常夢','夢見','鄰居托夢','時托夢','夢境','入夢','夢到那','夢到','噩夢',
                  '惡夢','實夢','夢中聞','夢鄉','夢話','夢樂透','做夢','作夢','托夢','夢醒',
                  '夢過','姊托夢','夢遊','夢魘','睡著','睡醒','睡覺','午睡','睡到',
                  '晚睡','熟睡','睡眠','昏睡','睡意','半睡半醒','睡死','打瞌睡','入睡'],
        "Bathroom": ['廁所'],
        "Window": ['窗戶', '窗簾','落地窗','氣窗','紙窗','窗台','玻璃窗','門窗','車窗',
                   '紗窗','小窗','外窗','櫥窗','鐵窗'],
        "Hallway": ['走廊', '黑長廊','長廊'],
        "Stairs": ['樓梯角','樓梯口','樓梯'],
        "School": ['學校', '教室', '宿舍','女生宿舍','男生宿舍','校園','夜校','校舍','住校',
                   '研究室','實驗室','大學','小學','中學','宿營','研究所'],
        "Outdoor": ['金寶山', '柏油路', '車站', '岔路','公路','小路','巡山','騎樓','佛光山','登山','金山',
                    '高山','山徑','武山','濱海','霞海','南海','海灘','斑馬線','黑森林','教會','墓地',
                    '墳墓區','墳墓','墓園','捷運站','車道','車場','公園','廟口','妻廟','祖師廟',
                    '桃園大廟','古廟','公廟','廟寺','大廟'],
        "Military Setting": ['部隊','查房','巡房','軍械室','國軍','軍區','軍卡車',
                     '美軍','軍方','軍隊','軍艦','軍機','軍工','行軍','軍校','軍旅','海軍',
                     '日軍','軍徽','單位','兵部','指揮部','旅部','營部',
                     '全營','營舍','營區','摸哨','哨站','站哨','撤哨','哨屋',
                     '接哨','上哨','下哨','外哨','哨點','哨所','內務櫃','戰車','甲車','碉堡']
    },
    "theqoo": {
        "Indoor (Domestic)": ['안방','침대','주방','방바닥','거실','독실','창고','숙소'],
        "Sleep_Dream": ['꿈꾸','꿈속','잠들','잠결','잠자','낮잠'],
        "Bathroom": ['화장실'],
        "Window": ['창문', '창가','창호지'],
        "Hallway": ['복도','현관'],
        "Stairs": ['계단'],
        "School": ['고등학교','교실','강의실','학교','초등학교','기숙사','대학'],
        "Outdoor": ['숲', '무덤','강원도','횡단보도','산길','사거리','뒷산',
                    '첩첩산중','교회','지하철','공원'],
        "Military Setting": ['중대','대대장','부대','초소','행보관']
    },
    "reddit": {
        "Indoor (Domestic)": ['room', 'bedroom', 'kitchen','house','playroom','storeroom',
                    'living room','livingroom','sunroom','guesthouse',
                    'warehouse','haunted house','townhouse','farmhouse','coachhouse',
                    'abandoned house','bedside'],
        "Sleep_Dream": ['dream','sleep paralysis','asleep','dreamed','dreaming','half-asleep',
                  'sleeper','sleepwalking','sleeping',
                  'half asleep','sleep','sleepwalk'],
        "Bathroom": ['bathroom', 'restroom', 'washroom','toilet'],
        "Window": ['window','windowsill'],
        "Hallway": ['hallway', 'corridor','hall','aisle','passage'],
        "Stairs": ['stair','stairwell','staircase','downstairs','upstairs','downstair','stairs'],
        "School": ['abandoned school','highschool','school', 'preschool',
                   'classroom', 'dormitory','dorm','elementary'],
        "Outdoor": ['church','ballroom','shophouse','lighthouse','dirtroad','roadside','backroad',
                    'railroad','crossroad','road','forest','woodland','woode','greenwood',
                    'graveyard','gravesite','cemetery','park','temple','subway','sidewalk'],
        "Military Setting": ['military', 'army','military base','soldier','navy']
    }
}

COLOR_GROUPS = {
    "ptt": {
        "White": ['白衣', '白色','全白','襲白','穿白','褲白鞋',
                  '白鞋','白點','白袍','白光','白線','青白','發白','白面','白霧',
                  '白布','白影'],
        "Red": [ '紅紙','紅裙','紅影','紅點','紅絲','染紅','紅字畫','紅布','紅褲',
                '發紅','紅線','紅光','血色'],
        "Black": ['黑影', '黑鬼','黑氣','黑色','黑青','烏黑','黑衣服','黑服',
                  '全黑','焦黑','黑衣','黑人','發黑','灰黑色','黑霧','黑煙','暗黑',
                  '黑面','黑衣人'],
        "Green": ['綠氣','綠霧','綠衣','綠光','綠綠','陣綠光','草綠服',
                  '變綠','青色'],
        "Blue": ['藍光','藍面','藏青色','鬼火'],
        "Yellow": ['黃衣','黃光'],
        "Pink": ['粉色'],
        "Brown": ['褐色','咖啡色'],
        "Grey": ['灰色','灰白色'],
        "Orange": ['橘色','金色'],
        "Purple": ['紫衣']
    },
    "theqoo": {
        "White": ['창백', '소복','흰색','하얀색','백발'],
        "Red": ['빨간색','피눈물'],
        "Black": ['흑색', '그림자',],
        "Blue": ['파란색','도깨비불','파란색','파랑색'],
        "Green": ['초록'],
        "Brown": ['갈색'],
        "Pink": ['핑크색'],
        "Grey": ['회백색','회색'],
        "Purple": ['연보라색']
    },
    "reddit": {
        "White": ['pale','whiteish','white','colorless'],
        "Red": ['redheade','reddish','red','bloody','bloodstained',
                'bloodied','blood'],
        "Pink": ['pink',],
        "Black": ['darken','dark','darkest','darker','darkness','blacken',
                  'blacker','blackness','black','pitch','shadow person','shadow figure',
                  'shadows','shadow','shadowy','shadowed'],
        "Grey": ['grey', 'gray','greyish','grayish'],
        "Green": ['green','greenish'],
        "Blue": ['blue','blueish'],
        "Purple": ['purple'],
        "Brown": ['brown','brownish']
    }
}

HAUNTING_METHOD_GROUPS = {
    "ptt": {
        "Sound": [ '敲打', '尖叫', '嚎啕大哭', '哭泣','叫喚','叫出','狂叫','大叫','大吼大叫',
                  '吠叫','喊叫','哀叫','哭叫','叫住','叫著','嚎叫','呼叫','叫喊','怪叫','吼叫',
                   '狗叫','叫醒','敲門','敲著','敲敲','敲擊','鬼哭','痛哭','哭醒','哭喊','大哭',
                   '爆哭','嘶吼','吼聲','狂吼','怒吼','斯吼','人聲','女聲','低聲','小聲','作聲','輕聲',
                   '大聲','門聲','鍵盤聲','聲音','聲調','高聲','出聲','聲響','彈珠聲','氣聲','鬼聲','鈴聲',
                   '男聲','雜聲','鈴鐺聲','銅鈴聲','翁翁聲','回音','童音','噪音','雜音','氣音','吹口哨','風聲',
                   '唱歌','歌聲','獰笑','發笑','咧嘴一笑','嘻笑','笑容','笑笑','笑著','狂笑','微笑',
                   '奸笑','大笑','冷笑'],
        "Touch": ['抓著','抓出去','抓走','抓痕','抓住','抓到','拉著','拉下去','拉腳','往下拉','猛拍',
                  '拍肩膀','拍打','觸碰','碰觸','碰到','觸摸','撫摸','手摸','摸到','摸摸','摸著','山風','海風',
                  '冷風','狂風','大風','微風','寒風','涼風','推開','推下去','推著','推下'],
        "Visual_Appearance": ['注視','出現', '站著','站立','看見','盯著','盯上','對視'],
        "Movement": ['消失','進房','靠窗','跟到','跟來','跟去','跟壓','壓床','壓著',
                     '上樓','下樓','墜樓','跳樓','上頂樓','跳海','落海','打開',
                     '壓制','鬼壓','壓到','被壓','壓住','人壓','壓過','擠壓',
                     '飄出','飄去','飄移','飄起','飄移','飄過','飄走','飄動','飄浮','飄著',
                     '飄進','穿越','穿門','穿梭','狂奔','奔跑','跑步','跑掉','飛奔',
                     '附著','依附','附身'],
    },
    "theqoo": {
        "Sound": ['휘파람','물소리','노랫소리','소리','목소리','발소리','소리치',
                  '울컥','울리','울음','외치','웃음'],
        "Touch": ['손길','바람','붙잡히','잡히','만지','어루만지','만지작','끌어안','끌려가',
                  '밀어내'],
        "Visual_Appearance": [ '바라보','돌아보','노려보','지켜보','내려다보','쳐다보','보이다',
                               '나타나','다가서','뒤돌아서'],
        "Movement": ['가위눌리','가위눌림','가위','따라오','누르','따라가','지나가','지나','지나치','사라지',
                     '움직이','뛰쳐나오','뛰어오','뛰어나가','씌이','빙의'],
    },
    "reddit": {
        "Sound": ['scream','screaming','voice','disembodied voice','disembodied','shrieking',
                   'shriek','crying','cry','breath','breathing','breathe','laughing','laugher',
                  'laughter','laugh','whistle','sound','sounding','shout','shouting','shouted'],
        "Touch": ['pull','tapping','tap','grab','touch','touching','push','slam','pushing','slamming'],
        "Visual_Appearance": [ 'watching','reappear','appear',
                               'standing','stare','watch','see','unseen','saw'],
        "Movement": ['move','disappear','disappearing','dissappeare','disappearance','following','follow',
                     'walking','walk','vanish','vanishing','float','floated','running','run',
                     'possessed','possess','possession'],
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# Unified lookup: domain → groups dict
# ─────────────────────────────────────────────────────────────────────────────
ALL_DOMAIN_GROUPS = {
    'space':           SPACE_GROUPS,
    'color':          COLOR_GROUPS,
    'haunting_method': HAUNTING_METHOD_GROUPS,
}

def get_all_keywords(platform):
    """Returns flat list of all thematic keywords for a platform."""
    all_kws = []
    for domain_groups in ALL_DOMAIN_GROUPS.values():
        if platform in domain_groups:
            for kws in domain_groups[platform].values():
                all_kws.extend(kws)
    return list(set(all_kws))

for p in ['ptt', 'theqoo', 'reddit']:
    kws = get_all_keywords(p)
    print(f'{p.upper()}: {len(kws)} unique thematic keywords')

---
## Part 2: Data Loading & Tokenisation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# UPDATE THESE PATHS to match your local directory structure
# ─────────────────────────────────────────────────────────────────────────────
RAW_DATA = Path(os.path.expanduser(
    '~/Downloads/ghost-story-gender-age-analysis/data/raw'
))

DATA_FILES = {
    'ptt':    RAW_DATA / 'new_ptt_experience_articles_all.csv',
    'reddit': RAW_DATA / 'reddit_merged_stories.csv',    
    'theqoo': RAW_DATA / 'theqoo_experience_articles_v2.csv',
}
TEXT_COL = 'content'

raw_texts = {}
for name, path in DATA_FILES.items():
    if path.exists():
        df = pd.read_csv(path)
        raw_texts[name] = df[TEXT_COL].dropna().astype(str).tolist()
        print(f'{name.upper()}: {len(raw_texts[name]):,} stories loaded.')
    else:
        print(f'[WARNING] {name.upper()}: file not found at {path}')
        raw_texts[name] = []

# If Reddit is split across two files, use this instead:
# reddit_paths = [
#     RAW_DATA / 'reddit_ghoststories_articles_all.csv',
#     RAW_DATA / 'reddit_paranormal_articles_all.csv',
# ]
# reddit_dfs = [pd.read_csv(p) for p in reddit_paths if p.exists()]
# raw_texts['reddit'] = pd.concat(reddit_dfs)[TEXT_COL].dropna().astype(str).tolist()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tokenisation functions (aligned with thesis_gender_ghost_analysis_v2.ipynb)
# ─────────────────────────────────────────────────────────────────────────────

_PROJ_ROOT = os.path.expanduser('~/Downloads/ghost-story-gender-age-analysis')

# ═══════════════════════════════════════════════════════════════════════════════
# STOPWORDS
# ═══════════════════════════════════════════════════════════════════════════════

def _load_file_stopwords(path):
    p = os.path.expanduser(path)
    if os.path.exists(p):
        with open(p, encoding='utf-8') as f:
            return {ln.strip() for ln in f if ln.strip() and not ln.startswith('#')}
    return set()

# --- English ---
STOPWORDS_EN = _load_file_stopwords(
    '~/Downloads/ghost-story-gender-age-analysis/data/raw/stopwords-en.txt'
) | {
    'like','just','really','actually','basically','literally','obviously',
    'maybe','probably','definitely','honestly','apparently','seriously',
}

# --- Korean ---
STOPWORDS_KO = (
    _load_file_stopwords('~/Downloads/ghost-story-gender-age-analysis/stopwords-ko.txt')
    | _load_file_stopwords('~/Downloads/ghost-story-gender-age-analysis/stopwords-ko_v2.txt')
    | {
        # Chosung abbreviations
        'ㅎㅎ','ㅋㅋ','ㅠㅠ','ㅜㅜ',
        # Morpheme / particle noise
        '아','음','응','야','헉','어','뭐','것','좀','네','아니',
        '그래','그럼','왜','어떻게','어디','언제','님','수','어요','나오','서',
        # Discourse connectors
        '그래서','그런데','그러면','하지만','그리고','또한','즉','왜냐하면','왜냐면',
        '어떤','이런','저런','아무','정말','진짜','매우','너무','아주','누구',
    }
)

# --- Chinese ---
def _build_stopwords_zh():
    words = set()
    try:
        import sys as _sys
        _tcsp_path = os.path.join(_PROJ_ROOT, 'TCSP-0.0.9')
        if _tcsp_path not in _sys.path:
            _sys.path.insert(0, _tcsp_path)
        from TCSP import read_stopwords_list
        words.update(read_stopwords_list())
    except (ImportError, Exception):
        pass
    # PTT boilerplate noise
    words.update({
        '未經授','權者','文章','商業','用途','分享','轉載',
        '嗨','哈','哈哈','呵呵','嘻嘻','嘿嘿','哎呀','啊呀',
        '哦','嗯','啊','喔','唉','哼','哇','呀','咦','哎哟',
        '恕','原文','手機','排版','請見諒','經驗文','真的','住',
        '發文','文章代碼','時間','看板','標題','作者',
    })
    # Baseline fallback
    words.update({
        '的','了','在','是','們',
        '和','與','或','但','而','也','都','很','就','到','從',
        '這','那','有','一','說','看','去','來',
        '什麼','怎麼','哪','誰','為什麼','因為','所以','雖然','如果',
    })
    return words

STOPWORDS_ZH = _build_stopwords_zh()

print(f"Stopwords loaded: EN={len(STOPWORDS_EN)} KO={len(STOPWORDS_KO)} ZH={len(STOPWORDS_ZH)}")

# ═══════════════════════════════════════════════════════════════════════════════
# CUSTOM VOCABULARY
# ═══════════════════════════════════════════════════════════════════════════════

# --- jieba (Chinese) ---
JIEBA_CUSTOM_WORDS = [
    # Ghost types
    '女鬼','男鬼','紅衣女鬼','魔神仔','好兄弟','水鬼','嬰靈',
    '鬼壓床','鬼打牆','陰陽眼','撞鬼','見鬼','鬼影幢幢',
    # Compound phenomena
    '靈異事件','超自然現象','靈魂出竅','農曆七月',
    '中元節','中元普渡','冤親債主','附身事件',
    # Practitioners
    '陰陽師','風水師傅','通靈師','驅魔師','乩童',
    # Locations
    '廢棄醫院','廢棄學校','廢棄旅館','民雄鬼屋','鬼屋',
    # Bodily sensations
    '雞皮疙瘩','毛骨悚然','鬼壓身',
    # Ritual objects
    '符咒','八卦鏡','桃木劍',
    # Taiwanese folk religion / ritual
    '觀落陰','地基主','拜地基主','孤魂野鬼','招魂術',
    '陰陽界','前世冤孽','靈媒師','通靈儀式','民間信仰',
    '清明節','普渡祭祀','祭祖祭拜','超渡儀式','法會','法事',
    # From original w2v notebook (sound / appearance keywords for LLR)
    '靈異','紅衣小女孩','山靈',
    '哭聲','口哨','腳步聲','低語','呢喃','耳語','笑聲','尖叫',
    '哭泣','呻吟','嘆息','呼吸聲','敲門聲','拍打聲','嚎啕大哭',
    '長頭髮','短頭髮','紅衣','白衣','黑影','白影','洋裝',
    '旗袍','收驚','軍徽','軍卡','守衛室','保健室',
]

# Load external custom words TXT if available
_custom_txt = os.path.join(_PROJ_ROOT, 'custom_words_ghost_analysis_traditional_chinese.txt')
if os.path.exists(_custom_txt):
    with open(_custom_txt, encoding='utf-8') as f:
        _ext = [ln.strip() for ln in f if ln.strip() and not ln.startswith('#')]
    JIEBA_CUSTOM_WORDS = list(dict.fromkeys(JIEBA_CUSTOM_WORDS + _ext))

if JIEBA_OK:
    for w in JIEBA_CUSTOM_WORDS:
        jieba.add_word(w)
    print(f"jieba: {len(JIEBA_CUSTOM_WORDS)} custom words registered.")

# --- spaCy protected phrases (English) ---
SPACY_PROTECTED_PHRASES = [
    'sleep paralysis', 'shadow person', 'shadow figure',
    'old hag syndrome', 'near death experience',
    'out of body experience', 'out-of-body experience',
    'half awake', 'half asleep', 'half-awake', 'half-asleep',
    'living room', 'school building', 'abandoned building',
    'abandoned house', 'abandoned school',
    'ouija board', 'spirit box', 'EVP session',
    'ghost hunting', 'ghost hunter', 'paranormal activity',
    'paranormal investigation', 'haunted location',
    'disembodied voice', 'cold touch', 'sage smudging',
    'urban exploration', 'abandoned mansion',
    # Additional from original w2v notebook
    'hat man', 'white lady',
]

if SPACY_OK:
    for phrase in SPACY_PROTECTED_PHRASES:
        nlp_en.tokenizer.add_special_case(phrase, [{"ORTH": phrase}])
    print(f"spaCy: {len(SPACY_PROTECTED_PHRASES)} protected phrases registered.")

# --- Kiwi custom words (Korean) ---
KIWI_CUSTOM_WORDS = [
    '처녀귀신','총각귀신','달걀귀신','손각시','지박령','부유령','동자귀신',
    '귀신들림','빨간마스크','화장실귀신','남자귀신','여자귀신','남귀신','여귀신',
    '귀신이야기','귀신소리','귀신공포','귀신출몰','몽달귀신','물귀신','머슴귀신',
    # From original w2v notebook
    '가위눌림','울음소리','발소리','속삭임','휘파람',
]

if KIWI_OK:
    for w in KIWI_CUSTOM_WORDS:
        kiwi.add_user_word(w, 'NNG')
    kiwi.add_user_word('소리치', 'VV')
    print(f"Kiwi: {len(KIWI_CUSTOM_WORDS) + 1} custom words registered.")

# ═══════════════════════════════════════════════════════════════════════════════
# ABBREVIATION / SLANG NORMALIZATION
# ═══════════════════════════════════════════════════════════════════════════════

AFFECT_ABBREVS = {
    'WTF':   'what the fuck',
    'OMG':   'oh my god',
    'OMFG':  'oh my fucking god',
    'FFS':   "for fuck's sake",
    'LMAO':  'laughing my ass off',
    'NGL':   'not gonna lie',
    'TBH':   'to be honest',
    'IDK':   "i don't know",
    'IYKYK': 'if you know you know',
    'DOA':   'dead on arrival',
    'RIP':   'rest in peace',
    'IWTD':  'i want to die',
}

KOREAN_CHOSUNG_MAP = {
    'ㄷㄷ':   '덜덜',
    'ㅎㄷㄷ': '후덜덜',
    'ㅂㄷ':   '벌벌',
    'ㄷㄱ':   '덜컹',
    'ㅆㅂ':   '씨발',
    'ㅁㅊ':   '미쳤어',
    'ㄱㅎ':   '공황',
    'ㅈㅅ':   '죄송',
    'ㄱㅅ':   '감사',
    'ㅇㅇ':   '응응',
    'ㄴㄴ':   '노노',
}

# ═══════════════════════════════════════════════════════════════════════════════
# POS TAG WHITELISTS (aligned with thesis)
# ═══════════════════════════════════════════════════════════════════════════════

KEEP_POS_EN  = {'NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN'}
KEEP_TAGS_KO = {'NNG', 'NNP', 'VV', 'VA', 'XR', 'MAG', 'NNB'}

# ═══════════════════════════════════════════════════════════════════════════════
# TOKENISATION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

# --- PTT (Chinese) ---
def tokenize_zh(text: str):
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
    if JIEBA_OK:
        tokens = jieba.lcut(text)
        return [w for w in tokens if len(w.strip()) >= 2 and w not in STOPWORDS_ZH]
    else:
        return [w for w in re.findall(r'[\u4e00-\u9fa5]{2,}', text) if w not in STOPWORDS_ZH]

# --- Reddit (English) ---
def tokenize_en(text: str):
    # Normalize affect abbreviations
    for abbr, expanded in AFFECT_ABBREVS.items():
        text = re.sub(rf'(?<!\w){re.escape(abbr)}(?!\w)', expanded, text, flags=re.IGNORECASE)

    text = text.lower()

    if SPACY_OK:
        doc = nlp_en(text)
        return [
            token.lemma_ for token in doc
            if token.pos_ in KEEP_POS_EN
            and token.lemma_ not in STOPWORDS_EN
            and len(token.lemma_) > 2
            and token.is_alpha
        ]
    else:
        # Regex fallback — preserve multi-word terms as underscored tokens
        replacements = {p: p.replace(' ', '_') for p in SPACY_PROTECTED_PHRASES if ' ' in p}
        for old, new in replacements.items():
            text = text.replace(old, new)
        tokens = re.findall(r"[a-z_']+", text)
        return [t for t in tokens if len(t) > 2 and t not in STOPWORDS_EN]

# --- Theqoo (Korean) ---
def tokenize_ko(text: str):
    text = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z0-9\s]', '', text)
    # Normalize 초성 abbreviations
    for raw, expanded in sorted(KOREAN_CHOSUNG_MAP.items(), key=lambda x: len(x[0]), reverse=True):
        text = text.replace(raw, expanded)
    if KIWI_OK:
        tokens = kiwi.tokenize(text)
        return [t.lemma for t in tokens
                if t.tag in KEEP_TAGS_KO and len(t.lemma) > 1 and t.lemma not in STOPWORDS_KO]
    else:
        return [w for w in re.findall(r'[가-힣]{2,}', text) if w not in STOPWORDS_KO]

TOKENIZERS = {
    'ptt':    tokenize_zh,
    'reddit': tokenize_en,
    'theqoo': tokenize_ko,
}

print('Tokenisers defined (aligned with thesis pipeline).')


In [ ]:
# Tokenise all platforms
tokenized = {}
for platform, texts in raw_texts.items():
    if not texts:
        print(f'[SKIP] {platform.upper()} — no texts.')
        tokenized[platform] = []
        continue
    print(f'Tokenising {platform.upper()} ({len(texts):,} stories)...')
    tokenized[platform] = [TOKENIZERS[platform](t) for t in texts]
    print(f'  Done. Sample: {tokenized[platform][0][:10]}')

---
## Part 3: Train Word2Vec Models

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# W2V hyperparameters — same as your original notebook
# vector_size=250, window=5, min_count=3, sg=1 (skip-gram), epochs=10
# ─────────────────────────────────────────────────────────────────────────────
W2V_PARAMS = dict(
    vector_size = 250,
    window      = 5,
    min_count   = 3,
    sg          = 1,
    workers     = 1,    # single-thread for full reproducibility
    epochs      = 10,
    seed        = 42,
)

w2v_models = {}
for platform, docs in tokenized.items():
    if not docs:
        print(f'[SKIP] {platform.upper()} — no tokenised data.')
        continue
    print(f'Training Word2Vec for {platform.upper()}...')
    model = Word2Vec(sentences=docs, **W2V_PARAMS)
    w2v_models[platform] = model
    print(f'  Vocabulary size: {len(model.wv):,}')

print('\nAll models trained.')

--- ## Part 4: Semantic Neighbourhood Query (Core Analysis)

For each thematic keyword, we query the top-10 most similar words in the
platform-specific Word2Vec model. Results are stored in a DataFrame with columns:
`platform`, `domain`, `category`, `keyword`, `rank`, `neighbour`, `cosine_sim`.

In [ ]:
def query_neighbourhood(model, keyword: str, topn: int = 10):
    """Return top-N similar words, or empty list if keyword not in vocab."""
    if keyword not in model.wv:
        return None  # not in vocabulary
    return model.wv.most_similar(keyword, topn=topn)


def _append_results(rows, model, platform, domain, category, kw, topn):
    """Helper to query one keyword and append rows."""
    result = query_neighbourhood(model, kw, topn=topn)
    if result is None:
        rows.append({
            'platform': platform, 'domain': domain,
            'category': category,
            'keyword': kw, 'rank': None,
            'neighbour': '[NOT IN VOCAB]', 'cosine_sim': None
        })
    else:
        for rank, (neighbour, sim) in enumerate(result, 1):
            rows.append({
                'platform': platform, 'domain': domain,
                'category': category,
                'keyword': kw, 'rank': rank,
                'neighbour': neighbour, 'cosine_sim': round(sim, 4)
            })


def run_thematic_query(platform: str, topn: int = 10):
    """
    For a given platform, iterate through all thematic keywords across
    all domains (space / color / haunting_method) and collect semantic
    neighbourhoods. Returns a DataFrame.
    """
    if platform not in w2v_models:
        print(f'[SKIP] No model for {platform.upper()}.')
        return pd.DataFrame()

    model = w2v_models[platform]
    rows = []

    for domain, domain_groups in ALL_DOMAIN_GROUPS.items():
        if platform not in domain_groups:
            continue
        for category, keywords in domain_groups[platform].items():
            for kw in keywords:
                _append_results(rows, model, platform, domain, category, kw, topn)

    return pd.DataFrame(rows)


# Run for all three platforms
results = {}
for platform in ['ptt', 'reddit', 'theqoo']:
    print(f'\nQuerying {platform.upper()}...')
    df = run_thematic_query(platform, topn=10)
    results[platform] = df
    found = df[df['neighbour'] != '[NOT IN VOCAB]']['keyword'].nunique()
    total = df['keyword'].nunique()
    print(f'  {found}/{total} keywords found in vocabulary.')

# Combine
results_all = pd.concat(results.values(), ignore_index=True)
print(f'\nTotal rows in combined results: {len(results_all):,}')

In [ ]:
# Save results to CSV
output_dir = Path('w2v_llr_results')
output_dir.mkdir(exist_ok=True)

for platform, df in results.items():
    out_path = output_dir / f'w2v_thematic_{platform}.csv'
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'Saved: {out_path}')

combined_path = output_dir / 'w2v_thematic_all_platforms.csv'
results_all.to_csv(combined_path, index=False, encoding='utf-8-sig')
print(f'Saved combined: {combined_path}')

--- ## Part 4b: Cosine Similarity Statistical Summary

Descriptive statistics of cosine similarity scores, grouped by **domain** and **category**, followed by inferential tests (Kruskal-Wallis *H* + pairwise Mann-Whitney *U* with BH-FDR correction).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cosine Similarity Descriptive Statistics
# ─────────────────────────────────────────────────────────────────────────────

valid_all = results_all[
    (results_all['neighbour'] != '[NOT IN VOCAB]') &
    (results_all['cosine_sim'].notna())
].copy()

# --- 1. Overall summary by platform × domain ---
print('=' * 70)
print('  COSINE SIMILARITY: Platform × Domain')
print('=' * 70)
summary_pd = valid_all.groupby(['platform', 'domain'])['cosine_sim'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(4)
print(summary_pd)
print()

# --- 2. Summary by platform × domain × category ---
print('=' * 70)
print('  COSINE SIMILARITY: Platform × Domain × Category')
print('=' * 70)
summary_pdc = valid_all.groupby(
    ['platform', 'domain', 'category']
)['cosine_sim'].agg(['count', 'mean', 'std', 'median']).round(4)
print(summary_pdc)
print()

# --- 3. Top-1 neighbour only (tightest semantic association) ---
top1 = valid_all[valid_all['rank'] == 1]
print('=' * 70)
print('  TOP-1 NEIGHBOUR COSINE SIMILARITY: Platform × Domain × Category')
print('=' * 70)
summary_top1 = top1.groupby(
    ['platform', 'domain', 'category']
)['cosine_sim'].agg(['count', 'mean', 'std', 'median']).round(4)
print(summary_top1)
print()

# --- 4. Box plot: cosine similarity distribution by domain per platform ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, platform in zip(axes, ['ptt', 'theqoo', 'reddit']):
    pdata = valid_all[valid_all['platform'] == platform]
    if pdata.empty:
        ax.set_title(f'{platform.upper()} (no data)')
        continue
    pdata.boxplot(
        column='cosine_sim', by='domain', ax=ax,
        patch_artist=True,
        boxprops=dict(facecolor='#d4e6f1'),
        medianprops=dict(color='#c0392b', linewidth=2),
    )
    ax.set_title(platform.upper(), fontsize=13)
    ax.set_xlabel('Domain')
    ax.set_ylabel('Cosine Similarity' if ax == axes[0] else '')
    ax.get_figure().suptitle('')

fig.suptitle(
    'Distribution of Cosine Similarity by Domain (all categories, top-10 neighbours)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig('w2v_llr_results/boxplot_cosine_by_domain.png', dpi=200, bbox_inches='tight')
print('Saved: w2v_llr_results/boxplot_cosine_by_domain.png')
plt.show()

# --- 5. Box plot: by category per platform × domain ---
for platform in ['ptt', 'theqoo', 'reddit']:
    pdata = valid_all[valid_all['platform'] == platform]
    if pdata.empty:
        continue
    domains = pdata['domain'].unique()
    fig, axes = plt.subplots(1, len(domains), figsize=(6 * len(domains), 5), sharey=True)
    if len(domains) == 1:
        axes = [axes]
    for ax, domain in zip(axes, domains):
        ddata = pdata[pdata['domain'] == domain]
        categories = ddata['category'].unique()
        data_list = [ddata[ddata['category'] == c]['cosine_sim'].dropna().values for c in categories]
        bp = ax.boxplot(
            data_list, labels=categories,
            patch_artist=True,
            boxprops=dict(facecolor='#d5f5e3'),
            medianprops=dict(color='#c0392b', linewidth=2),
        )
        ax.set_title(f'{domain.upper()}', fontsize=12)
        ax.set_xlabel('Category')
        ax.set_ylabel('Cosine Similarity' if ax == axes[0] else '')
        ax.tick_params(axis='x', rotation=45)
    fig.suptitle(
        f'{platform.upper()} — Cosine Similarity by Domain × Category',
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    fname = f'w2v_llr_results/boxplot_cosine_{platform}_by_domain_category.png'
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f'Saved: {fname}')
    plt.show()

# --- 6. Save summary tables to CSV ---
summary_pd.to_csv('w2v_llr_results/cosine_summary_platform_domain.csv', encoding='utf-8-sig')
summary_pdc.to_csv('w2v_llr_results/cosine_summary_platform_domain_category.csv', encoding='utf-8-sig')
summary_top1.to_csv('w2v_llr_results/cosine_summary_top1_platform_domain_category.csv', encoding='utf-8-sig')
print('\nAll summary CSVs saved to w2v_llr_results/')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Inferential Tests on Cosine Similarity
#   • Kruskal-Wallis H test (non-parametric one-way ANOVA)
#     — cosine similarity is bounded [−1, 1] and typically non-normal
#   • Pairwise Mann-Whitney U with BH-FDR correction (post-hoc)
#   • Rank-biserial correlation r as effect size
# ─────────────────────────────────────────────────────────────────────────────
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations

def rank_biserial(u, n1, n2):
    """Rank-biserial correlation r = 1 - 2U/(n1*n2)."""
    return 1 - (2 * u) / (n1 * n2)

def bh_fdr(p_values):
    """Benjamini-Hochberg FDR correction."""
    n = len(p_values)
    if n == 0:
        return []
    ranked = sorted(range(n), key=lambda i: p_values[i])
    adjusted = [0.0] * n
    for j, idx in enumerate(ranked):
        adjusted[idx] = p_values[idx] * n / (j + 1)
    # Enforce monotonicity (from largest rank down)
    for j in range(n - 2, -1, -1):
        idx = ranked[j]
        idx_next = ranked[j + 1]
        adjusted[idx] = min(adjusted[idx], adjusted[idx_next])
    return [min(p, 1.0) for p in adjusted]

# ── 1. Kruskal-Wallis: compare domains within each platform ──────────────
print('=' * 70)
print('  KRUSKAL-WALLIS: Cosine Similarity across Domains (per platform)')
print('=' * 70)

kw_rows = []
for platform in ['ptt', 'theqoo', 'reddit']:
    pdata = valid_all[valid_all['platform'] == platform]
    domains = sorted(pdata['domain'].unique())
    groups = [pdata[pdata['domain'] == d]['cosine_sim'].dropna().values for d in domains]
    groups = [g for g in groups if len(g) > 0]

    if len(groups) < 2:
        continue

    stat, p_val = kruskal(*groups)
    kw_rows.append({
        'platform': platform,
        'comparison': ' vs '.join(domains),
        'H': round(stat, 4),
        'p_value': p_val,
        'n_groups': len(groups),
        'total_n': sum(len(g) for g in groups),
    })
    print(f"  {platform.upper():8s}  H = {stat:8.4f}  p = {p_val:.6f}  (k = {len(groups)}, N = {sum(len(g) for g in groups)})")

kw_df = pd.DataFrame(kw_rows)
print()

# ── 2. Pairwise Mann-Whitney U with BH-FDR (post-hoc per platform) ───────
print('=' * 70)
print('  PAIRWISE MANN-WHITNEY U (BH-FDR) + RANK-BISERIAL r')
print('=' * 70)

pw_rows = []
for platform in ['ptt', 'theqoo', 'reddit']:
    pdata = valid_all[valid_all['platform'] == platform]
    domains = sorted(pdata['domain'].unique())

    for d1, d2 in combinations(domains, 2):
        g1 = pdata[pdata['domain'] == d1]['cosine_sim'].dropna().values
        g2 = pdata[pdata['domain'] == d2]['cosine_sim'].dropna().values
        if len(g1) < 2 or len(g2) < 2:
            continue
        u_stat, p_val = mannwhitneyu(g1, g2, alternative='two-sided')
        r_rb = rank_biserial(u_stat, len(g1), len(g2))
        pw_rows.append({
            'platform': platform,
            'group_1': d1, 'group_2': d2,
            'n1': len(g1), 'n2': len(g2),
            'U': round(u_stat, 2),
            'p_value': p_val,
            'rank_biserial_r': round(r_rb, 4),
        })

pw_df = pd.DataFrame(pw_rows)
if len(pw_df) > 0:
    pw_df['p_adj'] = bh_fdr(pw_df['p_value'].tolist())
    pw_df['sig'] = pw_df['p_adj'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    )
    print(pw_df.to_string(index=False))
else:
    print('  No valid pairwise comparisons.')
print()

# ── 3. Kruskal-Wallis: compare platforms within each domain ──────────────
print('=' * 70)
print('  KRUSKAL-WALLIS: Cosine Similarity across Platforms (per domain)')
print('=' * 70)

kw_domain_rows = []
for domain in sorted(valid_all['domain'].unique()):
    ddata = valid_all[valid_all['domain'] == domain]
    platforms = sorted(ddata['platform'].unique())
    groups = [ddata[ddata['platform'] == p]['cosine_sim'].dropna().values for p in platforms]
    groups = [g for g in groups if len(g) > 0]

    if len(groups) < 2:
        continue

    stat, p_val = kruskal(*groups)
    kw_domain_rows.append({
        'domain': domain,
        'comparison': ' vs '.join(platforms),
        'H': round(stat, 4),
        'p_value': p_val,
        'n_groups': len(groups),
        'total_n': sum(len(g) for g in groups),
    })
    print(f"  {domain:20s}  H = {stat:8.4f}  p = {p_val:.6f}  (k = {len(groups)}, N = {sum(len(g) for g in groups)})")

kw_domain_df = pd.DataFrame(kw_domain_rows)
print()

# ── 4. Pairwise Mann-Whitney U across platforms within each domain ────────
print('=' * 70)
print('  PAIRWISE MANN-WHITNEY U (BH-FDR) across Platforms (per domain)')
print('=' * 70)

pw_plat_rows = []
for domain in sorted(valid_all['domain'].unique()):
    ddata = valid_all[valid_all['domain'] == domain]
    platforms = sorted(ddata['platform'].unique())

    for p1, p2 in combinations(platforms, 2):
        g1 = ddata[ddata['platform'] == p1]['cosine_sim'].dropna().values
        g2 = ddata[ddata['platform'] == p2]['cosine_sim'].dropna().values
        if len(g1) < 2 or len(g2) < 2:
            continue
        u_stat, p_val = mannwhitneyu(g1, g2, alternative='two-sided')
        r_rb = rank_biserial(u_stat, len(g1), len(g2))
        pw_plat_rows.append({
            'domain': domain,
            'platform_1': p1, 'platform_2': p2,
            'n1': len(g1), 'n2': len(g2),
            'U': round(u_stat, 2),
            'p_value': p_val,
            'rank_biserial_r': round(r_rb, 4),
        })

pw_plat_df = pd.DataFrame(pw_plat_rows)
if len(pw_plat_df) > 0:
    pw_plat_df['p_adj'] = bh_fdr(pw_plat_df['p_value'].tolist())
    pw_plat_df['sig'] = pw_plat_df['p_adj'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    )
    print(pw_plat_df.to_string(index=False))
else:
    print('  No valid pairwise comparisons.')
print()

# ── 5. Save all test results ─────────────────────────────────────────────
kw_df.to_csv('w2v_llr_results/kruskal_wallis_domains_per_platform.csv', index=False, encoding='utf-8-sig')
if len(pw_df) > 0:
    pw_df.to_csv('w2v_llr_results/pairwise_mannwhitney_domains_per_platform.csv', index=False, encoding='utf-8-sig')
kw_domain_df.to_csv('w2v_llr_results/kruskal_wallis_platforms_per_domain.csv', index=False, encoding='utf-8-sig')
if len(pw_plat_df) > 0:
    pw_plat_df.to_csv('w2v_llr_results/pairwise_mannwhitney_platforms_per_domain.csv', index=False, encoding='utf-8-sig')
print('All inferential test results saved to w2v_llr_results/')

---
## Part 5: Formatted Output for Thesis Discussion

Print readable summaries per theme for each platform.  
These are the evidence tables you can quote directly in Ch. 5.

In [ ]:
def print_domain_summary(platform: str, domain: str, top_neighbours: int = 5):
    """
    Print a readable summary of semantic neighbourhoods for a given domain.
    Shows top-N neighbours per keyword, grouped by category.
    """
    df = results[platform]
    subset = df[(df['domain'] == domain) & (df['neighbour'] != '[NOT IN VOCAB]')]
    subset = subset[subset['rank'] <= top_neighbours]

    print(f'\n{"="*70}')
    print(f'  PLATFORM: {platform.upper()}  |  DOMAIN: {domain.upper()}')
    print(f'{"="*70}')

    for category in subset['category'].unique():
        c_data = subset[subset['category'] == category]
        if c_data.empty:
            continue
        print(f'\n  [{category}]')
        for kw, grp in c_data.groupby('keyword', sort=False):
            neighbours = grp[['neighbour', 'cosine_sim']].values.tolist()
            nb_str = ', '.join([f"{n} ({s:.3f})" for n, s in neighbours])
            print(f'    {kw:15s} → {nb_str}')


# --- Print all domains for all platforms ---
for platform in ['ptt', 'theqoo', 'reddit']:
    for domain in ['space', 'color', 'haunting_method']:
        if platform in results and not results[platform].empty:
            print_domain_summary(platform, domain, top_neighbours=5)

---
## Part 6: Vocabulary Coverage Check

Identify which LLR keywords are NOT in the Word2Vec vocabulary.  
These require either (a) concordance-level close reading, or (b) noting as low-frequency terms in the thesis.

In [ ]:
print('VOCABULARY COVERAGE REPORT')
print('='*50)
for platform in ['ptt', 'theqoo', 'reddit']:
    if platform not in results or results[platform].empty:
        continue
    df = results[platform]
    missing = df[df['neighbour'] == '[NOT IN VOCAB]'][['domain', 'category', 'keyword']].drop_duplicates()
    found = df[df['neighbour'] != '[NOT IN VOCAB]']['keyword'].nunique()
    total = df['keyword'].nunique()
    print(f'\n{platform.upper()}: {found}/{total} keywords in vocabulary')
    if not missing.empty:
        print('  Keywords NOT in vocab (need close reading):')
        for _, row in missing.iterrows():
            print(f'    [{row["domain"]} / {row["category"]}] {row["keyword"]}')

--- ## Part 7: Visualisation — Semantic Neighbourhood Heatmap

For each domain, platform, and category, show a heatmap of top-N keywords
and their closest neighbours with cosine similarity values.

In [ ]:
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
def plot_neighbourhood_heatmap(platform: str, domain: str, category: str,
                                top_keywords: int = 8, top_neighbours: int = 5,
                                figsize=(14, 6)):
    """
    Heatmap: rows = keywords, columns = top neighbours, cell = cosine similarity.
    """
    # Configure CJK-capable fonts for both Traditional Chinese and Korean.
    import matplotlib.font_manager as fm

    font_family_stack = []

    # Prefer local Noto TC if available (good for Traditional Chinese).
    tc_font_path = os.path.expanduser(
        '~/Downloads/ghost-story-gender-age-analysis/Noto_Sans_TC (1)/NotoSansTC-VariableFont_wght.ttf'
    )
    if os.path.exists(tc_font_path):
        fm.fontManager.addfont(tc_font_path)
        font_family_stack.append(fm.FontProperties(fname=tc_font_path).get_name())

    # Add platform/system fallback fonts to cover Korean glyphs as well.
    font_family_stack.extend([
        'AppleGothic',          # macOS Korean
        'Noto Sans CJK KR',     # Linux/installed CJK Korean
        'Malgun Gothic',        # Windows Korean
        'PingFang TC',          # macOS Traditional Chinese
        'Microsoft JhengHei',   # Windows Traditional Chinese
        'Arial Unicode MS',
        'DejaVu Sans',
    ])

    plt.rcParams['font.family'] = font_family_stack
    plt.rcParams['axes.unicode_minus'] = False

    df = results[platform]
    subset = df[
        (df['domain'] == domain) &
        (df['category'] == category) &
        (df['neighbour'] != '[NOT IN VOCAB]') &
        (df['rank'] <= top_neighbours)
    ]
    if subset.empty:
        print(f'[SKIP] No data for {platform}/{domain}/{category}')
        return

    # Pivot: keyword × rank → cosine_sim
    pivot = subset.pivot_table(index='keyword', columns='rank',
                               values='cosine_sim', aggfunc='first')
    # Keep only top_keywords by mean similarity
    pivot = pivot.head(top_keywords)

    # Label axes with neighbour words
    label_pivot = subset.pivot_table(index='keyword', columns='rank',
                                     values='neighbour', aggfunc='first')
    label_pivot = label_pivot.reindex(pivot.index)

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(pivot, annot=label_pivot, fmt='', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Cosine Similarity'})
    ax.set_title(
        f'{platform.upper()} | {domain.upper()} | {category}\n'
        f'Semantic neighbourhoods of thematic keywords',
        fontsize=12, pad=12
    )
    ax.set_xlabel('Rank in semantic neighbourhood')
    ax.set_ylabel('Keyword')
    plt.tight_layout()
    safe_cat = category.replace(' ', '_').replace('(', '').replace(')', '')
    fname = f'w2v_llr_results/heatmap_{platform}_{domain}_{safe_cat}.png'
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f'Saved: {fname}')
    plt.show()


# Generate heatmaps for all domain × category combinations
for platform in ['ptt', 'theqoo', 'reddit']:
    if platform not in results or results[platform].empty:
        continue
    for domain, domain_groups in ALL_DOMAIN_GROUPS.items():
        if platform not in domain_groups:
            continue
        for category in domain_groups[platform]:
            plot_neighbourhood_heatmap(platform, domain, category)

---
## Part 8: Cross-Platform Keyword Overlap

Check whether LLR-significant keywords from one platform appear in other platforms' vocabulary — and if so, what their semantic neighbourhood is there.

This supports cross-cultural comparison in the Discussion chapter.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-Platform Conceptual Equivalence Check
# Because each platform uses a different language, we cannot query the same
# token across models. Instead, we define conceptual equivalence groups and
# compare their semantic neighbourhoods side by side.
# ─────────────────────────────────────────────────────────────────────────────

CATEGORY_EQUIVALENTS = {
    'space_indoor_domestic': {
        'ptt':    ['房間', '房門','廚房','臥房','客房','房屋','雅房','書房',
                    '套房','茅房','小書房','睡房','閣樓','儲藏室','和室',
                    '紅眠床','大床','病床','彈簧床','床位','床沿','床邊'],
        'theqoo': ['안방','침대','주방','거실','독실','창고','숙소'],
        'reddit': ['bedroom', 'kitchen','house','playroom','storeroom',
                    'living room','loungeroom','sunroom','guesthouse',
                    'warehouse','townhouse','farmhouse','bedside'],
    },
    'space_sleep_dream': {
        'ptt':    ['夢見', '夢境','入夢','夢到','噩夢',
                  '惡夢','夢鄉','夢話','作夢','托夢','夢醒',
                  '夢過','夢遊','夢魘','睡著','睡醒','睡覺','午睡','睡到',
                  '晚睡','熟睡','睡眠','昏睡','睡意','半睡半醒','睡死','打瞌睡','入睡'],
        'theqoo': ['꿈속','잠결','낮잠'],
        'reddit': ['dream','asleep','dreaming','sleepwalking','sleeping','sleep','sleepwalk'],
    },
    'space_bathroom': {
        'ptt':    ['廁所'],
        'theqoo': ['화장실'],
        'reddit': ['bathroom', 'restroom', 'washroom','toilet'],
    },
    'space_window': {
        'ptt':    ['窗戶', '窗簾','落地窗','氣窗','窗台','玻璃窗','門窗','車窗',
                   '紗窗','小窗','外窗','櫥窗','鐵窗'],
        'theqoo': ['창문', '창가','창호지'],
        'reddit': ['window', 'windowsill',],
    },
    'space_hallway': {
        'ptt':    ['走廊',],
        'theqoo': ['복도','현관'],
        'reddit': ['hallway', 'corridor','hall','aisle','passage'],
    },
    'space_stairs': {
        'ptt':    ['樓梯', '樓梯口'],
        'theqoo': ['계단'],
        'reddit': ['stair','stairwell','staircase','downstairs','upstairs','downstair',],
    },
    'space_school': {
        'ptt':    ['學校', '教室', '宿舍','女生宿舍','男生宿舍','校園','夜校','校舍','住校',
                   '研究室','實驗室','大學','小學','中學','宿營','研究所'],
        'theqoo': ['고등학교','교실','강의실','학교','초등학교','기숙사','대학'],
        'reddit': ['highschool','school', 'preschool',
                   'classroom', 'dormitory','dorm','elementary'],
    },
    'space_outdoor': {
        'ptt':    ['金寶山', '柏油路', '車站', '岔路','公路','小路','巡山','騎樓','佛光山','登山','金山',
                    '高山','山徑','武山','濱海','霞海','南海','海灘','斑馬線','黑森林','教會','墓地',
                    '墳墓區','墳墓','墓園','捷運站','車道','車場','公園','廟口','祖師廟',
                    '古廟','公廟','大廟'],
        'theqoo': ['무덤','강원도','횡단보도','산길','사거리','뒷산','교회','지하철','공원'],
        'reddit': ['church','ballroom','lighthouse','roadside','backroad',
                    'railroad','crossroad','road','forest','woodland','woode','greenwood',
                    'graveyard','gravesite','cemetery','park','temple','subway','sidewalk'],
    },
    'space_military': {
        'ptt':    ['部隊','查房','巡房','軍械室','國軍','軍區','軍方','軍隊','軍艦','軍機','軍校','軍旅',
                   '軍徽','單位','兵部','指揮部','旅部','營部',
                     '全營','營舍','營區','摸哨','哨站','站哨','撤哨','接哨','上哨','下哨','外哨','哨點','哨所','內務櫃','戰車','甲車','碉堡'],
        'theqoo': ['중대','대대장','부대','초소','행보관'],
        'reddit': ['military', 'army','soldier','navy'],
    },

    'color_white': {
        'ptt':    ['白衣', '白色','全白','穿白','白鞋','白點','白袍','白光','白線','青白','發白','白面','白霧',
                  '白布','白影'],
        'theqoo': ['창백', '소복','흰색','하얀색','백발'],
        'reddit': ['pale','white','colorless'],
    },
    'color_red': {
        'ptt':    ['紅紙','紅裙','紅影','紅點','紅絲','染紅','紅布','紅褲',
                '發紅','紅線','紅光','血色'],
        'theqoo': ['빨간색'],
        'reddit': ['reddish','red','bloody','blood'],
    },
    'color_black': {
        'ptt':    ['黑影', '黑鬼','黑氣','黑色','黑青','烏黑','黑衣服','全黑',
                   '焦黑','黑衣','黑人','發黑','灰黑色','黑霧','黑煙','暗黑',
                  '黑面','黑衣人'],
        'theqoo': ['그림자'],
        'reddit': ['darken','dark','darker','darkness','blacken',
                  'blacker','blackness','black','pitch','shadow','shadowy','shadowed'],
    },
    'color_green': {
        'ptt':    ['綠氣','綠霧','綠衣','綠光','綠綠','草綠服',
                  '變綠','青色'],
        'theqoo': ['초록'],
        'reddit': ['green','greenish'],
    },
    'color_blue': {
        'ptt':    ['藍光','藍面','藏青色','鬼火'],
        'theqoo': ['도깨비불','파란색',],
        'reddit': ['blue','blueish'],
    },
    'color_pink': {
        'ptt':    ['粉色'],
        'theqoo': [ '핑크색'],
        'reddit': ['pink'],
    },
    'color_brown': {
        'ptt':    ['褐色', '咖啡色'],
        'theqoo': ['갈색'],
        'reddit': ['brown'],
    },
    'color_grey': {
        'ptt':    ['灰色', '灰白色'],
        'theqoo': ['회색'],
        'reddit': ['grey', 'gray','greyish','grayish'],
    },
    'color_purple': {
        'ptt':    ['紫衣'],
        'theqoo': ['보라색', '자주색'],
        'reddit': ['purple'],
    },

    'haunting_sound': {
        'ptt':    ['敲打', '尖叫', '嚎啕大哭', '哭泣','叫喚','狂叫','大叫','大吼大叫',
                  '吠叫','喊叫','哀叫','哭叫','叫住','嚎叫','呼叫','叫喊','怪叫','吼叫',
                   '狗叫','叫醒','敲門','敲著','敲敲','敲擊','鬼哭','痛哭','哭醒','哭喊','大哭',
                   '爆哭','嘶吼','吼聲','狂吼','怒吼','人聲','女聲','低聲','小聲','作聲','輕聲',
                   '大聲','門聲','鍵盤聲','聲音','聲調','高聲','出聲','聲響','彈珠聲','氣聲','鬼聲','鈴聲',
                   '男聲','雜聲','鈴鐺聲','回音','童音','噪音','雜音','氣音','吹口哨','風聲',
                   '唱歌','歌聲','獰笑','發笑','咧嘴一笑','嘻笑','笑容','笑笑','狂笑','微笑',
                   '奸笑','大笑','冷笑'],
        'theqoo': ['울음', '소리','휘파람','노랫소리','소리','목소리','발소리','울컥','웃음',],
        'reddit': ['scream','screaming','voice','disembodied','shriek','crying','breath','breathing','breathe','laughing',
                   'laughter','laugh','whistle','sound','sounding','shout','shouting',],
    },
    'haunting_touch': {
        'ptt':    ['抓著','抓出去','抓走','抓痕','抓住','抓到','拉著','拉下去','拉腳','往下拉','猛拍',
                  '拍肩膀','拍打','觸碰','碰觸','碰到','觸摸','撫摸','手摸','摸到','摸摸','摸著','山風','海風',
                  '冷風','狂風','大風','微風','寒風','涼風','推開','推下去','推著','推下'],
        'theqoo': ['바람'],
        'reddit': ['pull','tapping','tap','grab','touch','touching','push','slam','pushing','slamming'],
    },
    'haunting_visual_appearance': {
        'ptt':    ['注視','出現', '站立','看見','盯上','對視'],
        'theqoo': ['보이다'],
        'reddit': ['watching','reappear','standing','stare','watch','unseen',],
    },

    'haunting_movement': {
        'ptt':    ['消失','進房','靠窗','壓床','壓著',
                     '上樓','下樓','墜樓','跳樓','跳海','落海','打開',
                     '壓制','鬼壓','壓到','壓住','人壓','壓過','擠壓',
                     '飄出','飄去','飄移','飄起','飄過','飄動','飄浮','飄著',
                     '飄進','穿越','穿門','穿梭','狂奔','奔跑','跑步','跑掉','飛奔',
                     '附著','依附','附身'],
        'theqoo': ['가위눌림','가위','빙의'],
        'reddit': ['disappear','disappearance','follow',
                     'walking','walk','vanish','vanishing','float',
                     'running','possessed','possess','possession'],
    }
}


def cross_platform_category_to_df(category: str, topn=5):
    equiv = CATEGORY_EQUIVALENTS.get(category)
    if equiv is None:
        print(f'[ERROR] Unknown category: {category}')
        return pd.DataFrame()

    rows = []

    for platform, keywords in equiv.items():
        if platform not in w2v_models:
            continue

        model = w2v_models[platform]

        if isinstance(keywords, str):
            keywords = [keywords]

        for kw in keywords:
            if kw not in model.wv:
                continue

            neighbours = model.wv.most_similar(kw, topn=topn)

            for rank, (neighbor, score) in enumerate(neighbours, start=1):
                rows.append({
                    'Category': category,
                    'Platform': platform,
                    'Anchor': kw,
                    'Neighbor': neighbor,
                    'Similarity': score,
                    'Rank': rank
                })

    return pd.DataFrame(rows)


all_dfs = []

for category in CATEGORY_EQUIVALENTS:
    df_cat = cross_platform_category_to_df(category, topn=5)
    all_dfs.append(df_cat)

final_df = pd.concat(all_dfs, ignore_index=True)

output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)

# CSV（論文最常用）
csv_path = os.path.join(output_dir, "cross_platform_concepts.csv")
final_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

# JSON（之後做 graph / dashboard 很好用）
json_path = os.path.join(output_dir, "cross_platform_concepts.json")
final_df.to_json(json_path, orient='records', force_ascii=False)

print("Saved:", csv_path)
print("Saved:", json_path)


In [ ]:
summary_df = (
    final_df
    .groupby(['Category', 'Platform'])
    .agg(
        avg_similarity=('Similarity', 'mean'),
        max_similarity=('Similarity', 'max'),
        n_neighbors=('Neighbor', 'count')
    )
    .reset_index()
)

summary_path = os.path.join(output_dir, "cross_platform_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')

print("Saved:", summary_path)

pivot_df = summary_df.pivot(
    index='Category',
    columns='Platform',
    values='avg_similarity'
).fillna(0)

In [ ]:


output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)

heatmap_df = summary_df.pivot(
    index='Category',
    columns='Platform',
    values='avg_similarity'
).fillna(0)

plt.figure(figsize=(8, max(6, len(heatmap_df) * 0.45)))
im = plt.imshow(heatmap_df.values, aspect='auto')

plt.xticks(range(len(heatmap_df.columns)), heatmap_df.columns)
plt.yticks(range(len(heatmap_df.index)), heatmap_df.index)
plt.colorbar(im, label='Average Cosine Similarity')
plt.title("Cross-Platform Semantic Similarity Heatmap")
plt.tight_layout()

heatmap_path = os.path.join(output_dir, "cross_platform_similarity_heatmap.png")
plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
plt.show()

print("Saved:", heatmap_path)

In [ ]:
plot_df = summary_df.copy()

platform_order = ['ptt', 'theqoo', 'reddit']
plot_df['Platform'] = pd.Categorical(plot_df['Platform'], categories=platform_order, ordered=True)
plot_df = plot_df.sort_values(['Category', 'Platform'])

categories = plot_df['Category'].unique()
x = np.arange(len(categories))
width = 0.25

ptt_vals = [plot_df[(plot_df['Category'] == c) & (plot_df['Platform'] == 'ptt')]['avg_similarity'].values[0]
            if len(plot_df[(plot_df['Category'] == c) & (plot_df['Platform'] == 'ptt')]) > 0 else 0
            for c in categories]

theqoo_vals = [plot_df[(plot_df['Category'] == c) & (plot_df['Platform'] == 'theqoo')]['avg_similarity'].values[0]
               if len(plot_df[(plot_df['Category'] == c) & (plot_df['Platform'] == 'theqoo')]) > 0 else 0
               for c in categories]

reddit_vals = [plot_df[(plot_df['Category'] == c) & (plot_df['Platform'] == 'reddit')]['avg_similarity'].values[0]
               if len(plot_df[(plot_df['Category'] == c) & (plot_df['Platform'] == 'reddit')]) > 0 else 0
               for c in categories]

plt.figure(figsize=(12, max(5, len(categories) * 0.4)))
plt.bar(x - width, ptt_vals, width, label='PTT')
plt.bar(x, theqoo_vals, width, label='TheQoo')
plt.bar(x + width, reddit_vals, width, label='Reddit')

plt.xticks(x, categories, rotation=45, ha='right')
plt.ylabel("Average Cosine Similarity")
plt.title("Cross-Platform Category Similarity Comparison")
plt.legend()
plt.tight_layout()

bar_path = os.path.join(output_dir, "cross_platform_similarity_bar.png")
plt.savefig(bar_path, dpi=300, bbox_inches='tight')
plt.show()

print("Saved:", bar_path)

In [ ]:
def get_domain(cat):
    if cat.startswith("space_"):
        return "SPACE"
    elif cat.startswith("color_"):
        return "COLOR"
    elif cat.startswith("haunting_"):
        return "HAUNTING_METHOD"
    else:
        return "OTHER"

summary_df['Domain'] = summary_df['Category'].apply(get_domain)

for domain in summary_df['Domain'].unique():
    sub = summary_df[summary_df['Domain'] == domain].copy()

    heatmap_df = sub.pivot(
        index='Category',
        columns='Platform',
        values='avg_similarity'
    ).fillna(0)

    plt.figure(figsize=(8, max(4, len(heatmap_df) * 0.5)))
    im = plt.imshow(heatmap_df.values, aspect='auto')

    plt.xticks(range(len(heatmap_df.columns)), heatmap_df.columns)
    plt.yticks(range(len(heatmap_df.index)), heatmap_df.index)
    plt.colorbar(im, label='Average Cosine Similarity')
    plt.title(f"{domain} Semantic Similarity Heatmap")
    plt.tight_layout()

    out_path = os.path.join(output_dir, f"{domain.lower()}_similarity_heatmap.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()

    print("Saved:", out_path)

--- ## Part 9: Export Summary Table for Thesis

Produce a concise per-platform summary table showing:
- Domain | Category | Keyword | Top-3 Neighbours (with cosine similarity)

In [ ]:
def build_thesis_table(platform: str, top_neighbours: int = 15):
    df = results[platform]
    valid = df[(df['neighbour'] != '[NOT IN VOCAB]') & (df['rank'] <= top_neighbours)]

    # Aggregate top-N neighbours into a single string per keyword
    agg = valid.groupby(['domain', 'category', 'keyword']).apply(
        lambda g: '; '.join([f"{row['neighbour']} ({row['cosine_sim']:.3f})"
                             for _, row in g.sort_values('rank').iterrows()])
    ).reset_index(name='top_neighbours')

    agg.insert(0, 'platform', platform.upper())
    return agg


thesis_tables = []
for platform in ['ptt', 'theqoo', 'reddit']:
    if platform in results and not results[platform].empty:
        t = build_thesis_table(platform, top_neighbours=15)
        thesis_tables.append(t)

thesis_df = pd.concat(thesis_tables, ignore_index=True)

# Save
thesis_out = Path('w2v_llr_results/thesis_table_semantic_neighbourhoods.csv')
thesis_df.to_csv(thesis_out, index=False, encoding='utf-8-sig')
print(f'Thesis table saved: {thesis_out}')
print(f'Total rows: {len(thesis_df)}')

# Preview
thesis_df.head(20)

---
## Methodological Note for Thesis

The following paragraph can be adapted for inclusion in Chapter 3 (Methodology):

> To contextualise the statistically significant collocates identified through Log-Likelihood Ratio (LLR) analysis, we trained separate Word2Vec models (skip-gram; vector size = 250; window = 5; min_count = 3; epochs = 10) on the full raw story corpora for each platform (PTT: *n* stories; Theqoo: *n* stories; Reddit: *n* stories). Rather than querying the models with generic anchor terms, we queried the semantic neighbourhood of the exact keywords identified as LLR-significant (sig\_fdr ≥ *p* < 0.05) within each thematic category (space, color, haunting methods). This targeted querying strategy allows us to reconstruct the narrative contexts in which these statistically distinctive terms appear, providing qualitative enrichment for the Discussion chapter without introducing additional keyword selection bias. Words absent from the Word2Vec vocabulary (typically very low-frequency items) were noted as requiring concordance-level close reading.

---
*End of notebook.*